#### setting tracking uri and connecting to dagshub

In [6]:
import mlflow
import dagshub

mlflow.set_tracking_uri("https://dagshub.com/gbera23-dev/Machine-Learning.mlflow")

dagshub.init(repo_owner='gbera23-dev', repo_name='Machine-Learning', mlflow=True)


Initialized MLflow to track repo "gbera23-dev/Machine-Learning"

Repository gbera23-dev/Machine-Learning initialized!

#### loading baseline model to notebook

In [7]:
model = mlflow.sklearn.load_model("models:/house_prices_lr_baseline/1")

#### loading test data in

In [8]:
import pandas as pd

train_data_file_path = "house-prices-advanced-regression-techniques/test.csv"
dataset_df = pd.read_csv(train_data_file_path).copy(True)

#### apply same preprocessing to this data

In [9]:
import json

outsider_col_names = ['Id']
secondary_cols = []
for col in dataset_df.columns:
    if any(c == '2' for c in col) :
        secondary_cols.append(col)

# the last column is about second floor, not about secondary feature, so I will remove it from secondary_cols
secondary_cols.remove('2ndFlrSF')
for col in secondary_cols :
    outsider_col_names.append(col)

numerical_part_df = dataset_df.select_dtypes(include=['int64', 'float64'])

categorical_part_df = dataset_df.select_dtypes(exclude=['int64', 'float64'])

average_nonNull_entries = (dataset_df.count().sum() / dataset_df.count().size).__round__()

[outsider_col_names.append(i) for i in dataset_df.columns if dataset_df[i].count() <= average_nonNull_entries * (40 / 100)]

for col_name in outsider_col_names:
   if dataset_df.columns.str.contains(col_name).any():
       dataset_df = dataset_df.drop(col_name, axis=1, inplace=False)
numerical_part_df = dataset_df.select_dtypes(include=['int64', 'float64'])


num_rows = len(numerical_part_df)

numeric_columns_with_Nan_entries = []

for col in numerical_part_df.columns :
    if dataset_df[col].count() != num_rows :
        numeric_columns_with_Nan_entries.append(col)

dataset_df['LotFrontage'] = dataset_df['LotFrontage'].fillna(dataset_df['LotFrontage'].median())
dataset_df['MasVnrArea'] = dataset_df['MasVnrArea'].fillna(0)
dataset_df['GarageYrBlt'] = dataset_df['GarageYrBlt'].fillna(dataset_df['GarageYrBlt'].median())


categorical_part_df = dataset_df.select_dtypes(exclude=['int64', 'float64'])

categorical_columns_with_Nan_entries = []

for col in categorical_part_df.columns :
    if dataset_df[col].count() != num_rows :
        (categorical_columns_with_Nan_entries.append(col))

for col in categorical_columns_with_Nan_entries :
    dataset_df[col] = dataset_df[col].fillna("None")
dataset_df = pd.get_dummies(dataset_df, dtype=int, drop_first=True)

# In inference notebook
client = mlflow.tracking.MlflowClient()
client.download_artifacts(1, "train_columns.json", ".")
with open("train_columns.json") as f:
    train_columns = json.load(f)

dataset_df = dataset_df.reindex(columns=train_columns, fill_value=0)


TypeError: expected str, bytes or os.PathLike object, not int